# scRNA-seq integration: lung atlas

This notebook runs the v1.2.0 **linear autoencoder → optimal transport** pipeline. The published AnnData is downloaded from a stable Figshare file when it is not already present.

- Data: [Lung_atlas_public.h5ad](https://ndownloader.figshare.com/files/24539942)
- Benchmark context: [scIB metrics lung example](https://scib-metrics.readthedocs.io/en/latest/notebooks/lung_example.html)

Set `SCBIOT_TUTORIAL_DATA` to reuse a local data directory. Set `SCBIOT_TUTORIAL_MAX_CELLS=0` for all cells.


In [ ]:
from pathlib import Path
import os
import urllib.request
import numpy as np
import scanpy as sc
import scbiot as scb

RANDOM_STATE = 0
ROOT = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd())).resolve()
if ROOT.name == "R":
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("SCBIOT_TUTORIAL_DATA", ROOT / "inputs")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(filename, url):
    path = DATA_DIR / filename
    if path.exists():
        return path
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, partial)
    partial.replace(path)
    return path

def subsample(adata, default_max):
    # Raw is not needed here and can prevent indexed reads from backed sparse files.
    if getattr(adata, "isbacked", False) and adata.raw is not None:
        adata.raw = None
    max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", default_max))
    if max_cells > 0 and adata.n_obs > max_cells:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(adata.n_obs, max_cells, replace=False))
        return adata[keep].to_memory() if getattr(adata, "isbacked", False) else adata[keep].copy()
    return adata.to_memory() if getattr(adata, "isbacked", False) else adata.copy()

AE_EPOCHS = int(os.environ.get("SCBIOT_AE_EPOCHS", "30"))
USE_GPU = os.environ.get("SCBIOT_USE_GPU", "0") == "1"


In [ ]:
path = fetch("lung_atlas.h5ad", "https://ndownloader.figshare.com/files/24539942")
adata = subsample(sc.read_h5ad(path), 20_000)
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()
adata


## Integrate


In [ ]:
adata = scb.pp.autoencoder(
    adata,
    input_key="counts",
    out_key="X_ae",
    batch_key="batch",
    n_top_genes=2000,
    latent_dim=30,
    max_epochs=AE_EPOCHS,
    early_stop_patience=5,
    random_state=RANDOM_STATE,
)
adata, metrics = scb.ot.integrate(
    adata,
    obsm_key="X_ae",
    batch_key="batch",
    out_key="X_ot",
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
metrics


## Inspect the result


In [ ]:
sc.pp.neighbors(adata, use_rep="X_ot", random_state=RANDOM_STATE)
sc.tl.umap(adata, random_state=RANDOM_STATE)
sc.pl.umap(adata, color=["batch", "cell_type"])
